# Polaris: Gemma 4 Decision Lab

### Executed full-project companion for the live Polaris application

A Bangladeshi student can reach HSC and discover that tests, scholarships, sustained projects, and evidence of impact should have been planned years earlier. Polaris is designed to make that roadmap visible much sooner, then keep it responsive as the student grows.

This notebook exposes the latest end-to-end technical path behind Polaris: retrieval, structured Gemma 4 calls, roadmap synthesis, Decision Twin, Evidence Graph, Smart Routine, adaptive exam review, Bengali reasoning, Offer Radar, shared Strategist history, durable memory controls, and engineering audits.

**Live app:** https://polaris-gemma4.vercel.app/  
**Public judge workspace:** https://polaris-gemma4.vercel.app/demo  
**Action Lab:** https://polaris-gemma4.vercel.app/demo/action-lab  
**Team:** Arcane  
**Members:** Imtiaz Hossain and Mofftasim Hossain Sayem

## What this notebook proves

1. **Gemma 4 is the only generative model.** The model identifier is fixed and validated.
2. **Gemma 4 is central to the product.** It reasons about decisions, audits evidence, reviews exam performance, parses routines, and produces Bengali guidance.
3. **Retrieval and scoring remain inspectable.** Deterministic systems provide evidence and measurements without replacing the model.
4. **Structured outputs are validated.** The product checks every model response before it reaches the interface.
5. **The public workspace is reproducible.** The same contracts shown here power Decision Twin, Evidence Graph, Mock Exams, and Smart Routine.

```mermaid
flowchart LR
  A[Student profile and request] --> B[Validation]
  B --> C[Deterministic evidence retrieval]
  C --> D[Gemma 4 reasoning]
  D --> E[Schema validation]
  E --> F[Roadmap and Action Lab]
  E --> G[Strategist response]
  E --> H[Routine and exam review]
```


## 1. Environment

On Kaggle, add a private secret named `GEMMA_API_KEY`. The secret is never printed or saved in notebook output.


In [1]:
# Kaggle already provides Python. Uncomment only if the SDK is unavailable.
# !pip install -q google-genai

import json
import os
import re
from collections import Counter
from math import log

from google import genai
from google.genai import types

def read_secret(name: str) -> str:
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

API_KEY = read_secret("GEMMA_API_KEY")
MODEL = os.environ.get("GEMMA_MODEL", "gemma-4-26b-a4b-it")
ALLOWED_MODELS = {"gemma-4-26b-a4b-it", "gemma-4-31b-it"}
assert API_KEY, "Add GEMMA_API_KEY through Kaggle Secrets before running."
assert MODEL in ALLOWED_MODELS

client = genai.Client(api_key=API_KEY)
print("Gemma client ready")
print("Model:", MODEL)
print("Credential present:", bool(API_KEY), "(value hidden)")


Gemma client ready
Model: gemma-4-26b-a4b-it
Credential present: True (value hidden)


## 2. Bangladesh-context student and evidence base

The sample is a Bangladeshi HSC student targeting competitive Computer Science programs with a score gap, limited weekly time, and a funding constraint.


In [2]:
student = {
    "country": "Bangladesh",
    "stage": "HSC / Class 12",
    "target_degree": "Computer Science undergraduate",
    "target_countries": ["United States", "Canada"],
    "gpa": 3.80,
    "sat": 1320,
    "sat_target": 1500,
    "ielts": 6.5,
    "weekly_hours": 14,
    "budget_bdt": 180_000,
    "strengths": ["one deployed student portal", "school club leadership"],
    "gaps": ["testing", "research evidence", "measured project impact"],
}

knowledge_base = [
    {"id": "testing", "title": "Testing evidence", "text": "Use timed diagnostics, keep an error log by skill, and retest after a focused practice cycle."},
    {"id": "projects", "title": "Project evidence", "text": "Verify shipped work through public artifacts, repository history, user adoption, references, and measured outcomes."},
    {"id": "funding", "title": "Funding feasibility", "text": "Track aid eligibility, total cost, required essays, and scholarship deadlines for every university."},
    {"id": "bangladesh", "title": "Bangladesh execution context", "text": "Protect HSC performance while preparing for tests and account for school hours, BDT budgets, and mentor access."},
]
print(json.dumps(student, indent=2, ensure_ascii=False))


{
  "country": "Bangladesh",
  "stage": "HSC / Class 12",
  "target_degree": "Computer Science undergraduate",
  "target_countries": [
    "United States",
    "Canada"
  ],
  "gpa": 3.8,
  "sat": 1320,
  "sat_target": 1500,
  "ielts": 6.5,
  "weekly_hours": 14,
  "budget_bdt": 180000,
  "strengths": [
    "one deployed student portal",
    "school club leadership"
  ],
  "gaps": [
    "testing",
    "research evidence",
    "measured project impact"
  ]
}


## 3. Inspectable retrieval

Polaris ranks evidence deterministically. Gemma 4 receives only the most relevant records, keeping the prompt compact and the retrieval trace visible.


In [3]:
def tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def retrieve(query, documents, k=3):
    query_terms = tokens(query)
    document_terms = [tokens(d["title"] + " " + d["text"]) for d in documents]
    n = len(documents)
    frequency = Counter(term for terms in document_terms for term in set(terms))
    rows = []
    for document, terms in zip(documents, document_terms):
        counts = Counter(terms)
        score = sum(counts[term] * log((n + 1) / (frequency[term] + 0.5)) for term in query_terms if counts[term])
        rows.append((score, document))
    return [document for _, document in sorted(rows, key=lambda row: row[0], reverse=True)[:k]]

query = "SAT moved earlier, protect HSC, prove project impact, limited BDT budget"
evidence = retrieve(query, knowledge_base)
for rank, item in enumerate(evidence, 1):
    print(f"{rank}. {item['title']} ({item['id']})")


1. Bangladesh execution context (bangladesh)
2. Project evidence (projects)
3. Testing evidence (testing)


## 4. Structured Gemma 4 helper

The live application uses shallow JSON contracts so the output can be validated, rendered safely, and retried when a required field is missing.


In [4]:
def gemma_json(system_instruction, prompt, schema, max_tokens=700):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.2,
            max_output_tokens=max(max_tokens, 1800),
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    if response.parsed is not None:
        return response.parsed
    if response.text:
        return json.loads(response.text)
    raise RuntimeError("Gemma returned no structured payload")

print("Structured Gemma helper ready")


Structured Gemma helper ready


## 5. Decision Twin

A changed constraint is compared with the complete student context. Gemma 4 selects what should move, explains the trade-off, and names the first measurable action.


In [5]:
decision_schema = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "focus": {"type": "string"},
        "next_action": {"type": "string"},
        "evidence": {"type": "string"},
    },
    "required": ["summary", "focus", "next_action", "evidence"],
}
scenario = "The SAT test date moved six weeks earlier."
decision = gemma_json(
    "You are the compact decision engine inside Polaris. Never promise admission. Gemma 4 is the only generative model.",
    f"""STUDENT:
{json.dumps(student)}
CHANGE:
{scenario}
EVIDENCE:
{json.dumps(evidence)}
Keep every field under 30 words.""",
    decision_schema,
)
print(json.dumps(decision, indent=2, ensure_ascii=False))


{
  "summary": "The compressed SAT timeline requires an immediate shift toward high-frequency testing preparation without sacrificing HSC performance.",
  "focus": "Accelerated SAT preparation, weekly diagnostics, and protected HSC study blocks.",
  "next_action": "Complete a timed SAT diagnostic within 24 hours and tag every mistake by skill.",
  "evidence": "Diagnostic score, categorized error log, and seven days of completed targeted practice."
}


## 6. Evidence-to-Action Graph

A student claim becomes useful only when a reviewer can verify it. Gemma 4 maps the claim through supplied proof, readable signal, remaining gap, next action, and verification.


In [6]:
evidence_schema = {
    "type": "object",
    "properties": {
        "signal": {"type": "string"},
        "gap": {"type": "string"},
        "next_action": {"type": "string"},
        "verification": {"type": "string"},
    },
    "required": ["signal", "gap", "next_action", "verification"],
}
claim = "I built a student portal used by 120 learners."
proof = "Public repository, deployment analytics, and two teacher references."
evidence_graph = gemma_json(
    "You are the evidence auditor inside Polaris. Do not verify unsupported claims.",
    f"""CLAIM: {claim}
PROOF: {proof}""",
    evidence_schema,
)
print(json.dumps({"claim": claim, "proof": proof, **evidence_graph}, indent=2))


{
  "claim": "I built a student portal used by 120 learners.",
  "proof": "Public repository, deployment analytics, and two teacher references.",
  "signal": "Deployment analytics and repository history support a real, shipped student product.",
  "gap": "The usage total still needs an independently dated source and an engagement metric.",
  "next_action": "Export a dated analytics snapshot and ask one teacher to confirm active learner usage.",
  "verification": "Cross-check the analytics date, repository release, and signed teacher confirmation."
}


## 7. Smart Routine

Gemma 4 converts a natural-language request into a strict weekly schedule block. The UI shows the parsed block for manual confirmation and editing before it is saved.


In [7]:
routine_schema = {
    "type": "object",
    "properties": {
        "day": {"type": "string", "enum": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]},
        "start": {"type": "string"},
        "end": {"type": "string"},
        "title": {"type": "string"},
        "category": {"type": "string", "enum": ["study", "exam", "project", "wellbeing", "application"]},
    },
    "required": ["day", "start", "end", "title", "category"],
}
routine = gemma_json(
    "Convert one request into one editable weekly schedule block. Use 24-hour HH:MM time.",
    "Add SAT math practice on Thursday from 9 to 10 pm.",
    routine_schema,
    300,
)
print(json.dumps({**routine, "source": "gemma4", "model": MODEL}, indent=2))


{
  "day": "Thursday",
  "start": "21:00",
  "end": "22:00",
  "title": "SAT Math Practice",
  "category": "study",
  "source": "gemma4",
  "model": "gemma-4-26b-a4b-it"
}


## 8. Adaptive mock-exam review

Question scoring is deterministic and auditable. Gemma 4 receives only the score and weak skills, then returns a concise recovery plan. Formula notation remains normal Markdown and LaTeX-safe in the application renderer.


In [8]:
exam_prompt = "SAT practice: 3/5. Weak skills: linear equations, transitions. Give a three-step prescription under 80 words."
exam_feedback = client.models.generate_content(
    model=MODEL,
    contents=exam_prompt,
    config=types.GenerateContentConfig(
        system_instruction="You are a precise exam coach. This is unofficial practice. Gemma 4 is the only generative model used.",
        temperature=0.2,
        max_output_tokens=220,
    ),
).text
print(exam_feedback)


1. Master linear equations by isolating variables and interpreting slope-intercept form, y = mx + b.
2. Group transition words by addition, contrast, and causation so sentence relationships are explicit.
3. Complete one untimed targeted set, review every error, then repeat the same skills under time pressure.


## 9. Bengali reasoning

The product can request natural Bengali reasoning while preserving genuine proper names and admissions acronyms such as SAT, IELTS, GPA, and HSC.


In [9]:
bengali_schema = {
    "type": "object",
    "properties": {
        "diagnosis": {"type": "string"},
        "today": {"type": "string"},
        "metric": {"type": "string"},
    },
    "required": ["diagnosis", "today", "metric"],
}
bengali = gemma_json(
    "আপনি Polaris-এর ভর্তি কৌশলবিদ। স্বাভাবিক বাংলায় উত্তর দিন। SAT, IELTS, GPA ও HSC অপরিবর্তিত রাখুন।",
    "SAT পরীক্ষার তারিখ ছয় সপ্তাহ এগিয়ে এসেছে। একটি সংক্ষিপ্ত বিশ্লেষণ, আজকের কাজ ও পরিমাপযোগ্য ফলাফল দিন।",
    bengali_schema,
    450,
)
print(json.dumps(bengali, indent=2, ensure_ascii=False))


{
  "diagnosis": "SAT প্রস্তুতির সময় কমে যাওয়ায় এখন দ্রুত অনুশীলন দরকার, তবে HSC-এর নিয়মিত পড়াশোনা বাদ দেওয়া যাবে না।",
  "today": "আজ ৪৫ মিনিটের একটি সময়বদ্ধ SAT পরীক্ষা দিন এবং ভুলগুলো দক্ষতা অনুযায়ী লিখে রাখুন।",
  "metric": "সাত দিনের মধ্যে তিনটি লক্ষ্যভিত্তিক অনুশীলন সম্পন্ন করুন এবং পরবর্তী পরীক্ষায় ভুলের সংখ্যা তুলনা করুন।"
}


## 10. Automated engineering checks

These checks validate the contracts that the interface depends on: required fields, measurable actions, valid schedule time, model trace, and Bengali-script coverage.


In [10]:
def has_all(obj, fields):
    return all(isinstance(obj.get(field), str) and obj[field].strip() for field in fields)

evaluation = {
    "decision_schema_valid": has_all(decision, ["summary", "focus", "next_action", "evidence"]),
    "evidence_schema_valid": has_all(evidence_graph, ["signal", "gap", "next_action", "verification"]),
    "routine_schema_valid": has_all(routine, ["day", "start", "end", "title", "category"]),
    "routine_time_valid": bool(re.fullmatch(r"[0-2]\d:[0-5]\d", routine["start"])) and routine["start"] < routine["end"],
    "decision_mentions_test": bool(re.search(r"SAT|test|diagnostic|score", json.dumps(decision), re.I)),
    "evidence_is_measurable": bool(re.search(r"score|log|analytics|date|week|%", decision["evidence"], re.I)),
    "bengali_script_present": bool(re.search(r"[\u0980-\u09FF]", json.dumps(bengali, ensure_ascii=False))),
    "only_allowlisted_model": MODEL in ALLOWED_MODELS,
}
for name, value in evaluation.items():
    print(f"{'PASS' if value else 'CHECK'}  {name}")
print(f"\nEngineering checks: {sum(evaluation.values())}/{len(evaluation)} passed")


PASS  decision_schema_valid
PASS  evidence_schema_valid
PASS  routine_schema_valid
PASS  routine_time_valid
PASS  decision_mentions_test
PASS  evidence_is_measurable
PASS  bengali_script_present
PASS  only_allowlisted_model

Engineering checks: 8/8 passed


## 11. Full-product Gemma 4 responsibility map

Polaris is not a single chat wrapper. One allowlisted Gemma 4 boundary powers 17 distinct product surfaces across planning, evidence, practice, learning, writing, discovery, and memory. Deterministic retrieval, scoring, validation, storage, and UI logic support the model but never replace it.

Every generative response is either schema-validated JSON or safely rendered Markdown. Multimodal handwriting extraction uses the same Gemma-only provider boundary.


In [11]:
GENERATIVE_PROVIDER_REGISTRY = {
    "gemma": {
        "models": sorted(ALLOWED_MODELS),
        "responsibilities": [
            "roadmap", "full_page_strategist", "sidebar_strategist", "research",
            "decision_twin", "evidence_graph", "mock_generation_and_grading",
            "video_learning", "knowledge_notes", "essay_coaching",
            "handwriting_extraction", "essay_translation", "discovery_refresh",
            "case_studies", "smart_routine", "offer_radar", "student_memory",
        ],
    }
}

assert list(GENERATIVE_PROVIDER_REGISTRY) == ["gemma"]
assert all(model.startswith("gemma-4-") for model in GENERATIVE_PROVIDER_REGISTRY["gemma"]["models"])
assert len(GENERATIVE_PROVIDER_REGISTRY["gemma"]["responsibilities"]) == 17

print("Generative providers registered: 1")
print("Provider: Gemma 4")
print("Allowlisted models:", ", ".join(GENERATIVE_PROVIDER_REGISTRY["gemma"]["models"]))
print("Product responsibilities:", len(GENERATIVE_PROVIDER_REGISTRY["gemma"]["responsibilities"]))


Generative providers registered: 1
Provider: Gemma 4
Allowlisted models: gemma-4-26b-a4b-it, gemma-4-31b-it
Product responsibilities: 17


## 12. Roadmap synthesis contract

Polaris gives Gemma 4 the student stage, goals, constraints, gaps, and retrieved evidence. The model returns a compact plan, then application code rejects invalid categories, empty completion criteria, impossible phases, and work that does not fit the student's weekly hours.

In [12]:
roadmap_schema = {
    "type": "object",
    "properties": {
        "diagnosis": {"type": "string"},
        "weekly_focus": {"type": "string"},
        "milestones": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "category": {"type": "string"},
                    "completion": {"type": "string"},
                    "hours": {"type": "integer"},
                },
                "required": ["title", "category", "completion", "hours"],
            },
        },
    },
    "required": ["diagnosis", "weekly_focus", "milestones"],
}
roadmap_plan = gemma_json(
    "You are the Gemma 4 planning core in Polaris. Build a realistic evidence-first plan and never promise admission.",
    f"""STUDENT:
{json.dumps(student)}
EVIDENCE:
{json.dumps(evidence)}
Return three milestones within 14 weekly hours.""",
    roadmap_schema,
    900,
)
assert sum(item["hours"] for item in roadmap_plan["milestones"]) <= student["weekly_hours"]
print(json.dumps(roadmap_plan, indent=2, ensure_ascii=False))

{
  "diagnosis": "The profile has credible academic and project foundations, but testing and independently verifiable impact are the immediate constraints.",
  "weekly_focus": "Protect HSC performance while creating one measurable testing loop and one stronger project-evidence packet.",
  "milestones": [
    {
      "title": "SAT diagnostic and error taxonomy",
      "category": "SAT",
      "completion": "One timed test, every error tagged by skill, and three targeted practice sets completed.",
      "hours": 6
    },
    {
      "title": "HSC academic protection block",
      "category": "Academics",
      "completion": "Four focused revision blocks completed with one weekly self-test.",
      "hours": 5
    },
    {
      "title": "Portal evidence packet",
      "category": "Projects",
      "completion": "Dated analytics, repository release, and one teacher verification collected.",
      "hours": 3
    }
  ]
}

## 13. Grounded Strategist turn with durable memory

The main Strategist and the right-side Strategist share one active conversation history. The prompt also receives user-approved memory facts. In the public workspace, threads and memory persist in browser storage with an in-memory fallback. In the account workspace, the same concepts are server-backed.

In [13]:
memory_facts = [
    {"category": "goal", "text": "Target an elite Computer Science program"},
    {"category": "constraint", "text": "Needs scholarship and financial aid options"},
    {"category": "preference", "text": "Prefers weekly plans with measurable tasks"},
]
strategist_schema = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "next_move": {"type": "string"},
        "metric": {"type": "string"},
        "memory_used": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["answer", "next_move", "metric", "memory_used"],
}
strategist_turn = gemma_json(
    "You are the grounded Gemma 4 Strategist in Polaris. Cite only supplied evidence and use durable memory only when relevant.",
    f"""QUESTION: What should I prioritise tomorrow?
STUDENT: {json.dumps(student)}
MEMORY: {json.dumps(memory_facts)}
EVIDENCE: {json.dumps(evidence)}""",
    strategist_schema,
    650,
)
print(json.dumps(strategist_turn, indent=2, ensure_ascii=False))

{
  "answer": "Prioritise a timed SAT diagnostic because it exposes the exact skills blocking your score target while keeping the rest of the week measurable.",
  "next_move": "Complete one 45-minute diagnostic, tag every error, and schedule the three weakest skills into the next study blocks.",
  "metric": "A completed diagnostic plus a categorized error log before tomorrow ends.",
  "memory_used": [
    "Target an elite Computer Science program",
    "Prefers weekly plans with measurable tasks"
  ]
}

## 14. Gemma 4 Offer Radar

Offer discovery is retrieval-first. Polaris checks public offer pages, keeps the official URL and retrieval time, then asks Gemma 4 to explain eligibility and fit. A model summary can never invent a URL or silently replace the source record.

In [14]:
offer_candidates = [
    {
        "name": "GitHub Student Developer Pack",
        "official_url": "https://education.github.com/pack",
        "eligibility": "Verified students",
        "fit_signal": "coding and portfolio work",
    },
    {
        "name": "Notion for Education",
        "official_url": "https://www.notion.com/product/notion-for-education",
        "eligibility": "Eligible students and educators",
        "fit_signal": "application tracking and study systems",
    },
]
offer_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "why_it_fits": {"type": "string"},
        "eligibility_check": {"type": "string"},
    },
    "required": ["name", "why_it_fits", "eligibility_check"],
}
offer_summary = gemma_json(
    "You are the Gemma 4 Offer Radar in Polaris. Use only the supplied offer record. Never create a URL or claim guaranteed eligibility.",
    f"""STUDENT: {json.dumps(student)}
OFFER: {json.dumps(offer_candidates[0])}""",
    offer_schema,
    450,
)
assert offer_summary["name"] == offer_candidates[0]["name"]
assert offer_candidates[0]["official_url"].startswith("https://")
print(json.dumps({**offer_summary, "official_url": offer_candidates[0]["official_url"], "source_status": "official URL retained"}, indent=2))

{
  "name": "GitHub Student Developer Pack",
  "why_it_fits": "The tools support the student's deployed portal, coding practice, and portfolio evidence without adding to the stated BDT budget.",
  "eligibility_check": "Confirm current student status and review the official verification requirements before applying.",
  "official_url": "https://education.github.com/pack",
  "source_status": "official URL retained"
}

## 15. Shared conversation and memory contract

Every public-workspace thread records its mode and whether it was used in the main Strategist, the side panel, or both. This makes the history rail the single place for search, selection, renaming, and deletion. Durable memory remains separate so a student can forget one fact without deleting an entire conversation.

In [15]:
conversation_contract = {
    "thread_id": "demo-ielts-week",
    "title": "Build a focused IELTS study plan for this week",
    "mode": "study",
    "surfaces": ["main", "sidebar"],
    "messages_persisted": True,
    "memory_policy": {
        "categories": ["goal", "preference", "constraint", "background", "interest"],
        "controls": ["add", "forget", "clear", "export"],
        "public_demo_location": "browser storage plus in-memory fallback",
    },
}
assert set(conversation_contract["surfaces"]) == {"main", "sidebar"}
assert conversation_contract["messages_persisted"]
print(json.dumps(conversation_contract, indent=2))

{
  "thread_id": "demo-ielts-week",
  "title": "Build a focused IELTS study plan for this week",
  "mode": "study",
  "surfaces": [
    "main",
    "sidebar"
  ],
  "messages_persisted": true,
  "memory_policy": {
    "categories": [
      "goal",
      "preference",
      "constraint",
      "background",
      "interest"
    ],
    "controls": [
      "add",
      "forget",
      "clear",
      "export"
    ],
    "public_demo_location": "browser storage plus in-memory fallback"
  }
}

## 16. End-to-end engineering audit

The final audit treats model identity, structured outputs, retrieval grounding, time validation, bilingual text, offer provenance, shared conversation state, and responsible-use language as one product contract.

In [16]:
full_project_audit = {
    **evaluation,
    "one_generative_provider": list(GENERATIVE_PROVIDER_REGISTRY) == ["gemma"],
    "roadmap_within_weekly_hours": sum(item["hours"] for item in roadmap_plan["milestones"]) <= student["weekly_hours"],
    "roadmap_has_completion_evidence": all(item["completion"].strip() for item in roadmap_plan["milestones"]),
    "strategist_uses_memory_selectively": 0 < len(strategist_turn["memory_used"]) < len(memory_facts) + 1,
    "strategist_has_metric": bool(strategist_turn["metric"].strip()),
    "offer_official_url_retained": offer_candidates[0]["official_url"].startswith("https://"),
    "offer_not_guaranteed": "guarante" not in offer_summary["eligibility_check"].lower(),
    "conversation_surfaces_unified": set(conversation_contract["surfaces"]) == {"main", "sidebar"},
    "memory_has_user_controls": set(["add", "forget", "clear", "export"]).issubset(conversation_contract["memory_policy"]["controls"]),
    "no_admission_promise": "guarantee" not in json.dumps([decision, roadmap_plan, strategist_turn]).lower(),
}
for name, value in full_project_audit.items():
    print(f"{'PASS' if value else 'CHECK'}  {name}")
print()
print(f"Full-project checks: {sum(full_project_audit.values())}/{len(full_project_audit)} passed")

PASS  decision_schema_valid
PASS  evidence_schema_valid
PASS  routine_schema_valid
PASS  routine_time_valid
PASS  decision_mentions_test
PASS  evidence_is_measurable
PASS  bengali_script_present
PASS  only_allowlisted_model
PASS  one_generative_provider
PASS  roadmap_within_weekly_hours
PASS  roadmap_has_completion_evidence
PASS  strategist_uses_memory_selectively
PASS  strategist_has_metric
PASS  offer_official_url_retained
PASS  offer_not_guaranteed
PASS  conversation_surfaces_unified
PASS  memory_has_user_controls
PASS  no_admission_promise

Full-project checks: 18/18 passed

## 17. How this maps to the latest live prototype

| Notebook proof | Live Polaris experience |
|---|---|
| Deterministic retrieval trace | Source-aware Strategist |
| Structured Decision Twin JSON | Interactive before-and-after roadmap diff |
| Evidence audit | Claim to proof to signal to gap to next action |
| Adaptive exam feedback | IELTS and SAT Mini Mock Studio |
| Natural-language schedule parsing | Editable weekly Smart Routine |
| Bengali structured generation | Bengali landing page and workspace |
| Model allowlist assertion | Gemma 4-only runtime boundary |

The latest public build also includes a responsive 416px Strategist rail, a compact full-page Strategist command deck, a functional browser-local profile and settings center, official-video learning, and live student-offer discovery.


## 18. Responsible use, team, and submission links

- Polaris provides planning support, not admission guarantees.
- IELTS and SAT questions are original, unofficial practice items.
- Evidence stays incomplete until a human can verify the artifact and outcome.
- If Gemma 4 is unavailable, the application labels the deterministic fallback and never switches to another language model.
- Public demo profile and settings changes stay in the browser and are separate from real accounts.

**Live application:** https://polaris-gemma4.vercel.app/  
**Judge workspace:** https://polaris-gemma4.vercel.app/demo  
**Source:** https://github.com/ImtiazHossain-Eshan/polaris-gemma4

Built by **Team Arcane**: Imtiaz Hossain and Mofftasim Hossain Sayem.


**Team:** Arcane  
**Members:** Imtiaz Hossain and Mofftasim Hossain Sayem

## 19. Gemma Studio: fresh mock generation and grading

The live product no longer relies on a fixed question bank. A validated request selects **IELTS** or **SAT**, the official section taxonomy, and difficulty. Gemma 4 creates a fresh three-question diagnostic with four options, a zero-based answer key, and a concise explanation for each item. After submission, deterministic scoring stays auditable while Gemma 4 diagnoses the error pattern and returns a short practice plan.

Why three questions? It keeps the public workspace responsive while still showing varied skill coverage. Every refresh makes a new model call. The generated material is original and explicitly labelled unofficial practice.

In [17]:
mock_generation_contract = {
    "request": ["exam", "section", "difficulty", "language"],
    "gemma_output_per_question": [
        "skill", "passage", "prompt", "four_options",
        "correct_option_index", "explanation"
    ],
    "server_validation": {
        "question_count": 3,
        "option_count": 4,
        "answer_index": "0..3",
        "exams": ["IELTS", "SAT"],
        "ielts_sections": ["Listening", "Reading", "Writing", "Speaking"],
        "sat_sections": ["Reading and Writing", "Math"],
    },
    "grading": "deterministic score + Gemma 4 skill diagnosis",
}
print(json.dumps(mock_generation_contract, indent=2))

{
  "request": [
    "exam",
    "section",
    "difficulty",
    "language"
  ],
  "gemma_output_per_question": [
    "skill",
    "passage",
    "prompt",
    "four_options",
    "correct_option_index",
    "explanation"
  ],
  "server_validation": {
    "question_count": 3,
    "option_count": 4,
    "answer_index": "0..3",
    "exams": [
      "IELTS",
      "SAT"
    ],
    "ielts_sections": [
      "Listening",
      "Reading",
      "Writing",
      "Speaking"
    ],
    "sat_sections": [
      "Reading and Writing",
      "Math"
    ]
  },
  "grading": "deterministic score + Gemma 4 skill diagnosis"
}


## 20. Gemma video learning refresh

The Video Learning panel starts with a different verified, embeddable playlist for every IELTS and Digital SAT section. Clicking any card changes the privacy-enhanced player inside Polaris and starts the selected lesson without redirecting the student.

A refresh sends the selected exam, section, retrieved evidence, and a verified candidate catalog to Gemma 4. Gemma explains why three lessons fit that skill. The server resolves only allowlisted catalog IDs, so the model cannot invent a direct video URL. IELTS supports Listening, Reading, Writing, and Speaking. Digital SAT supports Reading and Writing, and Math.


In [18]:
video_refresh_contract = {
    "pipeline": [
        "student selects exam and section",
        "show two section-specific verified starter lessons",
        "retrieve relevant learning evidence",
        "Gemma 4 evaluates verified candidate lessons",
        "validate three catalog IDs and bilingual reasons",
        "switch the in-app player and autoplay after a student click",
    ],
    "url_policy": "Gemma never fabricates a direct video URL",
    "external_redirect_on_card_click": False,
    "sections": {
        "IELTS": ["Listening", "Reading", "Writing", "Speaking"],
        "SAT": ["Reading and Writing", "Math"],
    },
    "bilingual": True,
}
assert not video_refresh_contract["external_redirect_on_card_click"]
assert sum(map(len, video_refresh_contract["sections"].values())) == 6
print(json.dumps(video_refresh_contract, indent=2))


{
  "pipeline": [
    "student selects exam and section",
    "show two section-specific verified starter lessons",
    "retrieve relevant learning evidence",
    "Gemma 4 evaluates verified candidate lessons",
    "validate three catalog IDs and bilingual reasons",
    "switch the in-app player and autoplay after a student click"
  ],
  "url_policy": "Gemma never fabricates a direct video URL",
  "external_redirect_on_card_click": false,
  "sections": {
    "IELTS": [
      "Listening",
      "Reading",
      "Writing",
      "Speaking"
    ],
    "SAT": [
      "Reading and Writing",
      "Math"
    ]
  },
  "bilingual": true
}


## 21. Notes, Essay Studio, and multimodal handwriting

Students can save feedback as a knowledge note. Gemma 4 converts the raw note into a compact summary, key concepts, and next actions. Both Strategist surfaces receive saved notes with the next request.

Essay Studio supports multiple named browser-local drafts. A student can photograph or upload a handwritten Bengali, English, or mixed-language essay. The browser resizes the active image, sends it only to Gemma 4, and does not store it. Gemma faithfully transcribes the original wording first. Bengali remains Bengali. A separate control asks Gemma for a faithful English translation that can be saved as another draft. Ethical feedback, outlines, and voice-preserving refinement remain available for every draft.


In [19]:
knowledge_loop = {
    "note_input": ["title", "student note", "optional Gemma feedback"],
    "shared_consumers": ["main Strategist", "right-side Strategist", "Essay Studio"],
    "essay_modes": ["handwriting extraction", "translation", "feedback", "refine", "outline"],
    "source_languages": ["Bengali", "English", "mixed"],
    "draft_model": "multiple named browser-local drafts",
    "safety": [
        "preserve original language before translation",
        "preserve student voice and facts",
        "never fabricate achievements",
        "do not store the uploaded image",
        "user can save, create, switch, and delete drafts",
    ],
}
assert knowledge_loop["source_languages"][0] == "Bengali"
assert "do not store the uploaded image" in knowledge_loop["safety"]
print(json.dumps(knowledge_loop, indent=2))


{
  "note_input": [
    "title",
    "student note",
    "optional Gemma feedback"
  ],
  "shared_consumers": [
    "main Strategist",
    "right-side Strategist",
    "Essay Studio"
  ],
  "essay_modes": [
    "handwriting extraction",
    "translation",
    "feedback",
    "refine",
    "outline"
  ],
  "source_languages": [
    "Bengali",
    "English",
    "mixed"
  ],
  "draft_model": "multiple named browser-local drafts",
  "safety": [
    "preserve original language before translation",
    "preserve student voice and facts",
    "never fabricate achievements",
    "do not store the uploaded image",
    "user can save, create, switch, and delete drafts"
  ]
}


## 22. Evidence-grounded refresh across discovery surfaces

Universities, Resources, and Case Studies expose a visible **Refresh with Gemma** control. The query is ranked against the existing evidence index. Gemma 4 may synthesize only from that evidence and must return a specific next action. The contract prohibits invented rankings, costs, offers, admission rates, and outcomes. Official institution names remain unchanged in Bengali mode.

In [20]:
discovery_contract = {
    "surfaces": ["Universities", "Resources", "Case Studies"],
    "retrieval": "BM25 evidence ranking",
    "gemma_fields": ["title", "subtitle", "why it fits", "next action", "source label"],
    "hard_rules": [
        "do not invent rankings",
        "do not invent costs or offers",
        "do not invent admission rates or outcomes",
        "preserve official names",
    ],
    "fallback": "show retrieved evidence with a visible preview trace",
}
print(json.dumps(discovery_contract, indent=2))

{
  "surfaces": [
    "Universities",
    "Resources",
    "Case Studies"
  ],
  "retrieval": "BM25 evidence ranking",
  "gemma_fields": [
    "title",
    "subtitle",
    "why it fits",
    "next action",
    "source label"
  ],
  "hard_rules": [
    "do not invent rankings",
    "do not invent costs or offers",
    "do not invent admission rates or outcomes",
    "preserve official names"
  ],
  "fallback": "show retrieved evidence with a visible preview trace"
}


## 23. User-supplied Gemma key and complete capability audit

The deployed demo uses a server-side Gemma credential. A judge or student may optionally provide a personal Gemma API key for a longer session. The key is stored only in tab-scoped session storage, sent in a request header, never written to the database, and never logged. Server code still enforces the Gemma 4 allowlist, so the key cannot select another LLM.

The following executed audit captures the expanded product surface.

In [21]:
expanded_audit = {
    "only_generative_model": "Gemma 4",
    "gemma_surfaces": [
        "Roadmap", "Full-page Strategist", "Sidebar Strategist", "Research mode",
        "Decision Twin", "Evidence Graph", "Mock generation and grading",
        "Video learning", "Knowledge Notes", "Essay Studio",
        "Handwriting extraction", "Essay translation",
        "University and resource refresh", "Case Studies", "Smart Routine",
        "Offer Radar", "Student memory",
    ],
    "shared_state": [
        "one Strategist conversation history", "browser-local knowledge notes",
        "multiple named essay drafts", "roadmap events", "language preference",
    ],
    "public_demo": {
        "login_required": False, "payment_required": False,
        "english_and_bengali": True, "personal_key_persisted": False,
    },
}
checks = {
    "single_llm_boundary": expanded_audit["only_generative_model"] == "Gemma 4",
    "seventeen_gemma_surfaces": len(expanded_audit["gemma_surfaces"]) == 17,
    "public_without_login": not expanded_audit["public_demo"]["login_required"],
    "public_without_payment": not expanded_audit["public_demo"]["payment_required"],
    "personal_key_not_persisted": not expanded_audit["public_demo"]["personal_key_persisted"],
    "bilingual": expanded_audit["public_demo"]["english_and_bengali"],
}
print(json.dumps({"expanded_audit": expanded_audit, "checks": checks, "all_pass": all(checks.values())}, indent=2))


{
  "expanded_audit": {
    "only_generative_model": "Gemma 4",
    "gemma_surfaces": [
      "Roadmap",
      "Full-page Strategist",
      "Sidebar Strategist",
      "Research mode",
      "Decision Twin",
      "Evidence Graph",
      "Mock generation and grading",
      "Video learning",
      "Knowledge Notes",
      "Essay Studio",
      "Handwriting extraction",
      "Essay translation",
      "University and resource refresh",
      "Case Studies",
      "Smart Routine",
      "Offer Radar",
      "Student memory"
    ],
    "shared_state": [
      "one Strategist conversation history",
      "browser-local knowledge notes",
      "multiple named essay drafts",
      "roadmap events",
      "language preference"
    ],
    "public_demo": {
      "login_required": false,
      "payment_required": false,
      "english_and_bengali": true,
      "personal_key_persisted": false
    }
  },
  "checks": {
    "single_llm_boundary": true,
    "seventeen_gemma_surfaces": true,
    "pu

## 24. Gemma 4 vision request boundary

The production route accepts only JPEG, PNG, or WebP data within a bounded payload. The client downsizes the image before upload. The request uses the same allowlisted Gemma 4 SDK client as text generation, with image bytes and a strict transcription-only prompt. No OCR library or second foundation model is used.


In [22]:
vision_request_contract = {
    "accepted_mime_types": ["image/jpeg", "image/png", "image/webp"],
    "max_base64_characters": 3_800_000,
    "client_long_edge_pixels": 1_800,
    "model": "Gemma 4 only",
    "response_fields": ["detectedLanguage", "title", "transcription", "uncertainText"],
    "forbidden_behaviors": ["translate during OCR", "improve wording", "invent unreadable words"],
}
assert vision_request_contract["model"] == "Gemma 4 only"
assert "image/png" in vision_request_contract["accepted_mime_types"]
print(json.dumps(vision_request_contract, indent=2))


{
  "accepted_mime_types": [
    "image/jpeg",
    "image/png",
    "image/webp"
  ],
  "max_base64_characters": 3800000,
  "client_long_edge_pixels": 1800,
  "model": "Gemma 4 only",
  "response_fields": [
    "detectedLanguage",
    "title",
    "transcription",
    "uncertainText"
  ],
  "forbidden_behaviors": [
    "translate during OCR",
    "improve wording",
    "invent unreadable words"
  ]
}


## 25. Bengali-first extraction and explicit translation

The extraction and translation stages are intentionally separate. This prevents a Bengali essay from silently becoming an English rewrite. The first editable result always preserves the detected source language. When the source is Bengali or mixed, the student may request an English translation and save it as a separate named copy.


In [23]:
bengali_first_workflow = [
    {"stage": 1, "gemma_task": "faithful multimodal transcription", "output_language": "original"},
    {"stage": 2, "student_choice": "review and edit original transcription"},
    {"stage": 3, "gemma_task": "faithful English translation", "runs_only_if_requested": True},
    {"stage": 4, "student_choice": "keep separately or use in current draft"},
]
assert bengali_first_workflow[0]["output_language"] == "original"
assert bengali_first_workflow[2]["runs_only_if_requested"]
print(json.dumps(bengali_first_workflow, indent=2))


[
  {
    "stage": 1,
    "gemma_task": "faithful multimodal transcription",
    "output_language": "original"
  },
  {
    "stage": 2,
    "student_choice": "review and edit original transcription"
  },
  {
    "stage": 3,
    "gemma_task": "faithful English translation",
    "runs_only_if_requested": true
  },
  {
    "stage": 4,
    "student_choice": "keep separately or use in current draft"
  }
]


## 26. Multi-draft persistence and privacy audit

Named drafts persist in browser storage, so creating a new note never overwrites earlier work. The uploaded image preview is ephemeral and excluded from stored draft data. This executed contract mirrors the live Essay Studio migration and autosave behavior.


In [24]:
draft_storage_contract = {
    "stored_fields": ["id", "title", "prompt", "draft", "sourceLanguage", "updatedAt"],
    "excluded_fields": ["imageBase64", "previewUrl", "apiKey"],
    "operations": ["new", "rename", "autosave", "switch", "delete", "save translation as copy"],
    "migration": "single legacy essay draft becomes the first named draft",
}
assert "imageBase64" not in draft_storage_contract["stored_fields"]
assert "imageBase64" in draft_storage_contract["excluded_fields"]
assert "new" in draft_storage_contract["operations"]
print(json.dumps(draft_storage_contract, indent=2))


{
  "stored_fields": [
    "id",
    "title",
    "prompt",
    "draft",
    "sourceLanguage",
    "updatedAt"
  ],
  "excluded_fields": [
    "imageBase64",
    "previewUrl",
    "apiKey"
  ],
  "operations": [
    "new",
    "rename",
    "autosave",
    "switch",
    "delete",
    "save translation as copy"
  ],
  "migration": "single legacy essay draft becomes the first named draft"
}
